# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. Ответ: 20 10 Обьяснение: Потому что вначале мы иниицируем Series и даем ему  ярлыки в вид букв, далее обращаемся по этим ярлыкам и получаем 20. Второй вывод это у нас обращение по индексу так как в списке
2. Алматы-NaN Астана-NaN Шымкент-3.0, Потому что добавление известного числа неизвестной выводит NaN, а int+int=float. также сортировка по алфавиту среди ярлыков идет
3. (2, 2) - так как маска дает задачи выводить строго больше 1 по х, это 2 значения, 2х2 таблица(матрица)
4. Series DataFrame, Так как датафрейм это сбор сериес, вначале мы обращаемся именно к сериес, а потом обозначаем что обращение идет к датафрейму и внутри него сериес
5. f.index.tolist() - обращается к f где сейчас (2, 2), а из индексов валидны 1 и 2, и их нужно положить в список вывод [1, 2]. А второе это обращение по координатам так скажем будет 5

In [13]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [14]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [40]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [85]:
# 1
# Vopros 1
print(df.shape)
print("-" * 60)

# Vopros 2
print(df.dtypes)
print("-" * 60)

# Vopros 3
empty_days = df.shape[0] - df['delivery_days'].count()
print(empty_days)
print("-" * 60)

# Vopros 4
print(f"Уникальных городов в списке - {df['city'].nunique()}")
print(df['city'].unique())
print(f"Уникальных категорий в списке - {df['category'].nunique()}")
print(f"Уникальных статусов в списке - {df['status'].nunique()}")
print(f"Уникальных customer_id в списке - {df['customer_id'].nunique()}")
print("-" * 60)

# Vopros 5
print('Проблема 1. Я думаю что отсутсвие срока доставки, по понятным причинам')
print('Проблема 2. Разные написания городов могут привести к утере данных')

(5463, 8)
------------------------------------------------------------
order_id           int64
order_date        object
customer_id       object
city              object
category          object
amount           float64
delivery_days    float64
status            object
dtype: object
------------------------------------------------------------
160
------------------------------------------------------------
Уникальных городов в списке - 14
['Алматы' 'Астана' 'Караганда' 'Шымкент' 'алматы' 'Атырау' 'шымкент'
 'Shymkent' 'астана' 'Актобе' 'Astana ' 'АЛМАТЫ' ' Алматы' 'Almaty']
Уникальных категорий в списке - 5
Уникальных статусов в списке - 2
Уникальных customer_id в списке - 2003
------------------------------------------------------------
Проблема 1. Я думаю что отсутсвие срока доставки, по понятным причинам
Проблема 2. Разные написания городов могут привести к утере данных


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [62]:
# 2
filtered_1 = df.loc[(paid & not_test), 'amount']
# Среднее
mean_amount = filtered_1.mean()
print(mean_amount)
print("-" * 60)

# Медиана
median_amount = filtered_1.median()
print(median_amount)
print("-" * 60)

# Перцентиль 25 и 75
perc_25 = filtered_1.quantile(0.25)
perc_75 = filtered_1.quantile(0.75)
print(perc_25, perc_75)
print("-" * 60)

# Типичный чек
print(median_amount)
print("-" * 60)

#  Что честнее показать в отчёте, среднее или медиану, и почему?
print('Думаю что честнее будет показать медиану, хотя стоит отметь что без тестовых данных среднее тоже особо не врет')
print("-" * 60)

14672.391348704346
------------------------------------------------------------
10300.0
------------------------------------------------------------
5880.0 18690.0
------------------------------------------------------------
10300.0
------------------------------------------------------------
Думаю что честнее будет показать медиану, хотя стоит отметь что без тестовых данных среднее тоже особо не врет
------------------------------------------------------------


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [88]:
# 3
# Доля отменённых заказов: по всем заказам
total_canceled = df.loc[(not_test & (~paid)), 'amount']
total_orded = df.loc[(not_test), 'amount']
total_perc = (total_canceled.shape[0] / total_orded.shape[0]) * 100
print(f'Общая доля отмененных заказов - {total_perc:.2f} % БЕЗ ТЕСТОВЫХ ДАННЫХ')
print("-" * 60)

# в Шымкенте
canceled_shymkent = df.loc[(df['city'].isin(SH) & not_test & (~paid)), 'amount']
total_shymkent = df.loc[(df['city'].isin(SH) & not_test), 'amount']
canc_perc_shymkent = (canceled_shymkent.shape[0] / total_shymkent.shape[0]) * 100
print(f'Доля отмененных заказов в Шымкенте - {canc_perc_shymkent:.2f} % БЕЗ ТЕСТОВЫХ ДАННЫХ')
print("-" * 60)

# в Алматы
ALA = ['Алматы', 'алматы', 'АЛМАТЫ', ' Алматы', 'Almaty']
canceled_almaty = df.loc[(df['city'].isin(ALA) & not_test & (~paid)), 'amount']
total_almaty = df.loc[(df['city'].isin(ALA) & not_test), 'amount']
canc_perc_almaty = (canceled_almaty.shape[0] / total_almaty.shape[0]) * 100
print(f'Доля отмененных заказов в Алматы - {canc_perc_almaty:.2f} % БЕЗ ТЕСТОВЫХ ДАННЫХ')
print("-" * 60)

#  Какой город выделяется и во сколько раз?
print('Как видим в Шымкенте оменяют заказ в 2 раза больше чем в суммарной выборке, и почти в 3 раза больше чем в Алматы (тут про долю в процентах от своего же города)')

Общая доля отмененных заказов - 10.24 % БЕЗ ТЕСТОВЫХ ДАННЫХ
------------------------------------------------------------
Доля отмененных заказов в Шымкенте - 20.40 % БЕЗ ТЕСТОВЫХ ДАННЫХ
------------------------------------------------------------
Доля отмененных заказов в Алматы - 7.87 % БЕЗ ТЕСТОВЫХ ДАННЫХ
------------------------------------------------------------
Как видим в Шымкенте оменяют заказ в 2 раза больше чем в суммарной выборке, и почти в 3 раза больше чем в Алматы (тут про долю в процентах от своего же города)


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [114]:
# 4
# Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы
orders_month_city = pd.DataFrame({
    'city': ['Алматы', 'Шымкент'],
    'september': [df.loc[((df['city'].isin(ALA)) & (paid) & (not_test) & (sep)), 'amount'].sum(), df.loc[((df['city'].isin(SH)) & (paid) & (not_test) & (sep)), 'amount'].sum()],
    'october': [df.loc[((df['city'].isin(ALA)) & (paid) & (not_test) & (octo)), 'amount'].sum(), df.loc[((df['city'].isin(SH)) & (paid) & (not_test) & (octo)), 'amount'].sum()],
})
print(orders_month_city)

# изменение в %
ala_change = ((orders_month_city['october'][0] - orders_month_city['september'][0]) / orders_month_city['september'][0]) * 100
sh_change = ((orders_month_city['october'][1] - orders_month_city['september'][1]) / orders_month_city['september'][1]) * 100
print(f'Изменение оплаченых заказов в Алмате по сравнению с сентябрем {ala_change:.2f}%')
print(f'Изменение оплаченых заказов в Шымкенте по сравнению с сентябрем {sh_change:.2f}%')

# Где падение сильнее и во сколько раз? Что сказать менеджеру?
print('В Шымкенте силное падение на оплаченные заказы на 18,3 процента, Нужно детальнее исследовать вопрос касательно этого')

      city   september     october
0   Алматы  15523940.0  14643020.0
1  Шымкент   4949800.0   4044010.0
Изменение оплаченых заказов в Алмате по сравнению с сентябрем -5.67%
Изменение оплаченых заказов в Шымкенте по сравнению с сентябрем -18.30%
В Шымкенте силное падение на оплаченные заказы на 18,3 процента, Нужно детальнее исследовать вопрос касательно этого


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [119]:
# 5
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [143]:
# 6
df['customer_id'].value_counts() # C01818 самый частый покупатель

# Покажите все заказы этого клиента (loc с маской, столбцы дата, город, сумма, статус).
most_resently_customer = df.loc[(df['customer_id'] == 'C01818'), ['order_date', 'city', 'amount', 'status']]
print(most_resently_customer)
print('-' * 60)

# Есть ли среди его заказов повторяющиеся order_id?
print(most_resently_customer.value_counts().head(1))
print('-' * 60)

#  Что это значит для выручки, которую мы считали весь урок?
print('То что дубли будут разбавлять среднюю цену')
print('-' * 60)


            order_date       city   amount   status
57    2026-09-01 15:00    Шымкент   7350.0  оплачен
779   2026-09-09 18:00    Шымкент  45270.0  оплачен
1371  2026-09-15 21:00     Алматы   8890.0  оплачен
1375  2026-09-15 21:00     Алматы   8890.0  оплачен
2195  2026-09-24 18:00     Астана   1490.0  оплачен
2617  2026-09-29 09:00     Астана  22170.0  оплачен
2776  2026-09-30 21:00     Алматы  20180.0  оплачен
3417  2026-10-08 14:00  Караганда   4330.0  оплачен
4279  2026-10-18 08:00     Астана  13600.0  оплачен
4610  2026-10-21 19:00     Алматы  21990.0  оплачен
4778  2026-10-23 18:00     Алматы  11140.0  оплачен
------------------------------------------------------------
order_date        city    amount  status 
2026-09-15 21:00  Алматы  8890.0  оплачен    2
Name: count, dtype: int64
------------------------------------------------------------
То что дубли будут разбавлять среднюю цену
------------------------------------------------------------


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8